## Divident based investing

In [1]:
import pandas as pd 
import numpy as np 
import yfinance as yf 
import math
from scipy import stats

In [2]:
tickers= pd.read_csv('top_50_indian_stocks.csv')
tickers.head()

,Ticker,Company Name
0,RELIANCE.NS,Reliance Industries
1,TCS.NS,Tata Consultancy Services
2,HDFCBANK.NS,HDFC Bank
3,INFY.NS,Infosys
4,ICICIBANK.NS,ICICI Bank


## Weighted scoring model for finding the final score


In [10]:
import pandas as pd
import numpy as np
import yfinance as yf

def create_dividend_df(tickers):

    columns = [
        "Ticker",
        "Dividend_Yield(%)",
        "Dividend Rate",
        "Payout Ratio(%)",
        "5 Year Dividend Growth Rate(%)",
        "Earning Growth(%)"
    ]

    dividend_df = pd.DataFrame(columns=columns)

    for stock in tickers:
        ticker = yf.Ticker(stock)
        info = ticker.info

        divident_yield = (
            info.get("dividendYield", np.nan) * 100
            if info.get("dividendYield") is not None
            else np.nan
        )

        divident_rate = (
            info.get("dividendRate", np.nan)
            if info.get("dividendRate") is not None
            else np.nan
        )

        payout_ratio = (
            info.get("payoutRatio", np.nan) * 100
            if info.get("payoutRatio") is not None
            else np.nan
        )

        five_year_dividend_growth_rate = (
            info.get("fiveYearAvgDividendYield", np.nan) * 100
            if info.get("fiveYearAvgDividendYield") is not None
            else np.nan
        )

        earning_growth = (
            info.get("earningsGrowth", np.nan) * 100
            if info.get("earningsGrowth") is not None
            else np.nan
        )

        dividend_df.loc[len(dividend_df)] = [
            stock,
            divident_yield,
            divident_rate,
            payout_ratio,
            five_year_dividend_growth_rate,
            earning_growth
        ]

    numeric_cols = [
        "Dividend_Yield(%)",
        "Dividend Rate",
        "Payout Ratio(%)",
        "5 Year Dividend Growth Rate(%)",
        "Earning Growth(%)"
    ]

    for col in numeric_cols:

        if col == "Payout Ratio(%)":
            dividend_df[col + "_Normalized"] = 1 - (
                (dividend_df[col] - dividend_df[col].min()) /
                (dividend_df[col].max() - dividend_df[col].min())
            )

        else:
            dividend_df[col + "_Normalized"] = (
                (dividend_df[col] - dividend_df[col].min()) /
                (dividend_df[col].max() - dividend_df[col].min())
            )

    return dividend_df

In [11]:
tickers_list = tickers["Ticker"].tolist()
dividend_df = create_dividend_df(tickers_list)
dividend_df

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: TATAMOTORS.NS"}}}


,Ticker,Dividend_Yield(%),Dividend Rate,Payout Ratio(%),5 Year Dividend Growth Rate(%),Earning Growth(%),Dividend_Yield(%)_Normalized,Dividend Rate_Normalized,Payout Ratio(%)_Normalized,5 Year Dividend Growth Rate(%)_Normalized,Earning Growth(%)_Normalized
0,RELIANCE.NS,45.0,6.0,9.21,44.0,-12.6,0.049517,0.02981,0.903083,0.047146,0.068275
1,TCS.NS,549.0,124.0,46.32,149.0,12.2,0.658213,0.669377,0.512575,0.177419,0.09128
2,HDFCBANK.NS,175.0,13.0,21.76,102.0,7.5,0.206522,0.067751,0.77102,0.119107,0.08692
3,INFY.NS,431.0,50.0,64.65,240.0,11.8,0.5157,0.268293,0.319689,0.290323,0.090909
4,ICICIBANK.NS,88.0,11.0,14.71,99.0,8.4,0.101449,0.056911,0.845207,0.115385,0.087755
5,HINDUNILVR.NS,204.0,44.0,95.03,158.0,21.4,0.241546,0.235772,0.0,0.188586,0.099814
6,SBIN.NS,180.0,17.35,17.44,150.0,-3.1,0.21256,0.091328,0.816479,0.17866,0.077087
7,BAJFINANCE.NS,59.0,5.4,14.42,24.0,21.4,0.066425,0.026558,0.848258,0.022333,0.099814
8,BHARTIARTL.NS,87.0,16.0,36.060002,80.0,-34.0,0.100242,0.084011,0.620541,0.091811,0.048423
9,ITC.NS,558.0,16.0,86.92,372.0,-72.7,0.669082,0.084011,0.085341,0.454094,0.012523


In [12]:
weights={
    "Dividend_Yield(%)_Normalized": 0.3,
    "Dividend Rate_Normalized": 0.2,
    "Payout Ratio(%)_Normalized": 0.2,
    "5 Year Dividend Growth Rate(%)_Normalized": 0.2,
    "Earning Growth(%)_Normalized": 0.1
}

dividend_df["Dividend_Score"] = dividend_df[[col for col in weights.keys()]].mul(list(weights.values())).sum(axis=1)
dividend_df

,Ticker,Dividend_Yield(%),Dividend Rate,Payout Ratio(%),5 Year Dividend Growth Rate(%),Earning Growth(%),Dividend_Yield(%)_Normalized,Dividend Rate_Normalized,Payout Ratio(%)_Normalized,5 Year Dividend Growth Rate(%)_Normalized,Earning Growth(%)_Normalized,Dividend_Score
0,RELIANCE.NS,45.0,6.0,9.21,44.0,-12.6,0.049517,0.02981,0.903083,0.047146,0.068275,0.217691
1,TCS.NS,549.0,124.0,46.32,149.0,12.2,0.658213,0.669377,0.512575,0.177419,0.09128,0.478466
2,HDFCBANK.NS,175.0,13.0,21.76,102.0,7.5,0.206522,0.067751,0.77102,0.119107,0.08692,0.262224
3,INFY.NS,431.0,50.0,64.65,240.0,11.8,0.5157,0.268293,0.319689,0.290323,0.090909,0.339462
4,ICICIBANK.NS,88.0,11.0,14.71,99.0,8.4,0.101449,0.056911,0.845207,0.115385,0.087755,0.242711
5,HINDUNILVR.NS,204.0,44.0,95.03,158.0,21.4,0.241546,0.235772,0.0,0.188586,0.099814,0.167317
6,SBIN.NS,180.0,17.35,17.44,150.0,-3.1,0.21256,0.091328,0.816479,0.17866,0.077087,0.28877
7,BAJFINANCE.NS,59.0,5.4,14.42,24.0,21.4,0.066425,0.026558,0.848258,0.022333,0.099814,0.209339
8,BHARTIARTL.NS,87.0,16.0,36.060002,80.0,-34.0,0.100242,0.084011,0.620541,0.091811,0.048423,0.194187
9,ITC.NS,558.0,16.0,86.92,372.0,-72.7,0.669082,0.084011,0.085341,0.454094,0.012523,0.326666


In [14]:
dividend_df = dividend_df.sort_values(by="Dividend_Score", ascending=False)
dividend_df.head(10)

,Ticker,Dividend_Yield(%),Dividend Rate,Payout Ratio(%),5 Year Dividend Growth Rate(%),Earning Growth(%),Dividend_Yield(%)_Normalized,Dividend Rate_Normalized,Payout Ratio(%)_Normalized,5 Year Dividend Growth Rate(%)_Normalized,Earning Growth(%)_Normalized,Dividend_Score
45,IOC.NS,713.0,10.0,32.709998,669.0,78.1,0.85628,0.051491,0.655793,0.822581,0.152412,0.578098
31,BPCL.NS,671.0,20.0,37.2,540.0,28.0,0.805556,0.105691,0.608545,0.662531,0.105937,0.527614
20,ONGC.NS,697.0,18.5,41.0,511.0,47.8,0.836957,0.097561,0.568557,0.626551,0.124304,0.522051
13,HCLTECH.NS,811.0,96.0,88.01,357.0,-0.2,0.974638,0.517615,0.073871,0.435484,0.079777,0.505763
30,COALINDIA.NS,480.0,22.0,52.32,812.0,12.9,0.574879,0.116531,0.449437,1.0,0.091929,0.49485
42,HEROMOTOCO.NS,377.0,185.0,61.04,313.0,25.7,0.450483,1.0,0.357677,0.380893,0.103803,0.493239
1,TCS.NS,549.0,124.0,46.32,149.0,12.2,0.658213,0.669377,0.512575,0.177419,0.09128,0.478466
22,POWERGRID.NS,422.0,12.25,44.75,474.0,9.6,0.504831,0.063686,0.529096,0.580645,0.088868,0.395021
17,WIPRO.NS,832.0,17.0,87.58,151.0,-1.6,1.0,0.089431,0.078396,0.179901,0.078479,0.377393
29,MARUTI.NS,107.0,140.0,28.91,80.0,-6.4,0.124396,0.756098,0.69578,0.091811,0.074026,0.353459
